# NB04 · 数据消融：Success = f(N)

| | |
|---|---|
| **目标** | 画出你的第一条数据缩放曲线，得到「边际数据价值」的第一个实测点——MDV 的起点 |
| **前置** | NB02（训练管道通）、NB03（评测协议） |
| **预计耗时** | 半天人工 + 8–16h 机器（建议云上并行） |
| **产出物** | `results/NB04.json` + 缩放曲线图 |
| **通过标准** | 曲线每点 ≥2 seeds 且带 CI；能给出「每 25 条 episode 值几个百分点」 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
import subprocess, sys, itertools
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nbutils

SUBSETS = [25, 50, 100, 206]      # pusht 共 206 条；换数据集时改
SEEDS   = [0, 1]
STEPS   = 50_000                  # 消融可用比 NB02 短的预算，但所有组必须一致
DRY_RUN = True                    # 先 True 打印命令核对，再 False 实跑
PROTOCOL = nbutils.latest("NB03")
print("评测协议：", PROTOCOL)

In [ ]:
# 子集构造验证：用 episodes= 参数取前 N 条（更严谨的做法是随机抽样并固定抽样 seed——想想为什么）
ds25 = nbutils.load_lerobot_dataset("lerobot/pusht", episodes=range(25))
print("episodes in subset:", ds25.num_episodes)
assert ds25.num_episodes == 25

In [ ]:
# 训练矩阵：4 sizes × 2 seeds = 8 个 run。云上可并行；本地串行要一夜。
# LeRobot 的子集训练参数名随版本不同（--dataset.episodes= 或需要先物化子集数据集），照 --help 核对。
for size, seed in itertools.product(SUBSETS, SEEDS):
    out = f"outputs/nb04/n{size}_s{seed}"
    cmd = (f"{sys.executable} -m lerobot.scripts.train "
           f"--policy.type=diffusion --env.type=pusht --dataset.repo_id=lerobot/pusht "
           f"--dataset.episodes='[0..{size-1}]' --seed={seed} --steps={STEPS} --output_dir={out}")
    print(cmd)
    if not DRY_RUN:
        subprocess.run(cmd, shell=True, check=True)

In [ ]:
# 评测矩阵 → 结果表。eval 输出接不上就人肉填 MEASURED（数字必须进账本）。
MEASURED = {   # (size, seed): success_rate
    # (25, 0): 0.12,  (25, 1): 0.08,
    # (50, 0): ...,
}
assert MEASURED, "跑完 eval 后把 8 个数字填进来"
table = {s: [MEASURED[(s, seed)] for seed in SEEDS if (s, seed) in MEASURED] for s in SUBSETS}
table

In [ ]:
# 缩放曲线 + 边际价值
n_eval = nbutils.latest("NB03")["n_episodes"]
means = np.array([np.mean(table[s]) for s in SUBSETS])
plt.figure(figsize=(7, 4))
for s in SUBSETS:
    for v in table[s]:
        lo, hi = nbutils.wilson_ci(int(v * n_eval), n_eval)
        plt.plot([s, s], [lo, hi], color="gray", alpha=0.5)
plt.plot(SUBSETS, means, "o-")
plt.xlabel("N training episodes"); plt.ylabel("success rate"); plt.title("Success = f(N), pusht + Diffusion Policy")
plt.savefig("results/NB04_scaling.png", dpi=120, bbox_inches="tight"); plt.show()

for i in range(1, len(SUBSETS)):
    dn = SUBSETS[i] - SUBSETS[i-1]
    dp = (means[i] - means[i-1]) * 100
    print(f"第 {SUBSETS[i-1]}→{SUBSETS[i]} 条：边际价值 {dp/dn:+.2f} pp/episode")

nbutils.log_result("NB04", {"subsets": SUBSETS, "seeds": SEEDS, "steps": STEPS,
                            "table": {str(k): v for k, v in table.items()}})

## 分析

1. **曲线形状**：对数增长？已饱和？还在陡坡上？「再加 200 条同分布数据」值得吗——你的证据？
2. **定价句式**（练到脱口而出）：「在这个任务上，第 100–200 条 episode 每条约值 X 个百分点，按采集成本 Y 元/条，每个百分点的成功率成本是 Z 元。」
3. **方差警告**：同 size 两个 seed 差多少？如果 seed 间差异 > 相邻 size 间差异，你的曲线还可信吗？该加 seed 还是加 eval 次数？（用 NB03 的框架回答。）
4. **这张图的去处**：这是《数据资产审计》报告的实证脚注、也是 observatory/experiments 的第一条记录。commit 它。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
